# U9c · TabFM (modelo fundacional tabular) vs. XGBoost afinado

> **⚠️ Datos SINTÉTICOS.** Ningún paciente real. · **Necesita GPU** (en Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU*).

En este cuaderno ponemos a prueba una idea que hace un año parecía ciencia ficción: un **modelo fundacional para tablas** que predice **sin entrenar y sin ajustar hiperparámetros** (*zero-shot*), y que aun así compite con un **XGBoost bien afinado**.

Usaremos **TabFM** (Google Research, 2026) y lo compararemos, sobre nuestros datos clínicos, con un XGBoost al que sí dedicaremos el trabajo habitual de ajuste.

**El plan:**
1. Partimos de los datos **sucios** (`pacientes_sucio.csv`) y los **limpiamos** (como en la U2). Veremos que *esto hay que hacerlo igual*: TabFM te ahorra el modelado, **no** la limpieza.
2. Entrenamos y **afinamos un XGBoost** (el campeón clásico de datos tabulares).
3. Lanzamos **TabFM en zero-shot** (sin entrenar nada).
4. Los comparamos **cara a cara** con las métricas de la U3.

> **Licencia:** el **código** de TabFM es Apache-2.0; sus **pesos** tienen licencia **no comercial**. Para docencia e investigación, perfecto; para un producto, revisa la licencia.


## 1. Generamos los datos del curso

Como siempre, la primera celda crea los CSV sintéticos si no están en la sesión. No hay que descargar nada.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

## 2. El punto de partida: datos **sucios**

Cargamos `pacientes_sucio.csv`, la versión con los problemas de calidad que ya conoces de la U2 (unidades mezcladas, texto en campos numéricos, outliers imposibles, categorías inconsistentes, duplicados). Miramos su estado.


In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings("ignore")

sucio = pd.read_csv("pacientes_sucio.csv")
print("Forma:", sucio.shape)
print("\nTipos de algunas columnas (fíjate que son 'object' = texto, no números):")
print(sucio[["glucemia", "colesterol_total", "ta_diastolica"]].dtypes)
sucio[["paciente_id","edad","sexo","glucemia","colesterol_total","ta_diastolica","tabaquismo"]].head()

**Lo que vemos:** columnas que *deberían* ser numéricas (glucemia, colesterol, tensión) están guardadas como **texto**, porque contienen cosas como `"desconocido"`, `"N/D"` o números con coma (glucemia en mmol/L). Además hay edades imposibles, sexos escritos de mil formas y filas duplicadas.

> **⚠️ Ni TabFM ni XGBoost pueden con esto tal cual.** Un modelo, por muy fundacional que sea, **no adivina** que un `"desconocido"` es un valor ausente ni que una `edad = 250` es un error. La limpieza sigue siendo **nuestra** responsabilidad. Vamos a ello.


## 3. Limpieza (la misma idea que en la U2)

Aplicamos una limpieza compacta: normalizar categorías, convertir a número (recuperando la glucemia en mmol/L), anular outliers imposibles, imputar ausentes y quitar duplicados. Cada bloque va comentado.


In [ ]:
df = sucio.copy()

# id sin espacios y sin duplicados
df["paciente_id"] = df["paciente_id"].astype(str).str.strip()
df = df.drop_duplicates()

# sexo -> M / F
df["sexo"] = df["sexo"].astype(str).str.strip().str.lower().map(
    lambda s: "M" if s in ["m","masculino","h","hombre"] else ("F" if s in ["f","femenino","mujer"] else np.nan))

# tabaquismo -> nunca / ex / activo
tab = {"nunca":"nunca","no fumador":"nunca","ex":"ex","exfumador":"ex","ex-fumador":"ex",
       "activo":"activo","fumador":"activo","si":"activo"}
df["tabaquismo"] = df["tabaquismo"].astype(str).str.strip().str.lower().map(lambda s: tab.get(s, np.nan))

# glucemia: coma->punto, a número; los valores bajos estaban en mmol/L -> *18 a mg/dL
g = pd.to_numeric(df["glucemia"].astype(str).str.replace(",", "."), errors="coerce")
df["glucemia"] = g.where(g > 30, g * 18)

# columnas con texto ("desconocido","N/D") -> número (lo no convertible queda como ausente)
df["colesterol_total"] = pd.to_numeric(df["colesterol_total"], errors="coerce")
df["ta_diastolica"]   = pd.to_numeric(df["ta_diastolica"], errors="coerce")

# outliers imposibles -> ausente
df.loc[(df.edad < 0) | (df.edad > 110), "edad"] = np.nan
df.loc[(df.ta_sistolica <= 0) | (df.ta_sistolica > 260), "ta_sistolica"] = np.nan
df.loc[(df.imc < 10) | (df.imc > 70), "imc"] = np.nan

# imputación simple: mediana (números) / moda (categorías)
num = ["edad","imc","ta_sistolica","ta_diastolica","glucemia","colesterol_total","hdl"]
for c in num: df[c] = df[c].fillna(df[c].median())
for c in ["sexo","tabaquismo"]: df[c] = df[c].fillna(df[c].mode()[0])

print("Tras limpiar:", df.shape, "| valores ausentes:", int(df.isna().sum().sum()))
df[["edad","sexo","glucemia","colesterol_total","ta_diastolica","tabaquismo"]].head()

**Lo que vemos:** la tabla ha quedado **limpia y numérica**, sin ausentes ni duplicados. Este es el trabajo que **ningún** modelo hace por ti.

> **💡 Idea clave.** TabFM elimina el **entrenamiento** y el **ajuste de hiperparámetros**, pero **no** el análisis y la limpieza de datos. En proyectos reales, esa parte sigue siendo la mitad del trabajo (justo lo que vimos en el caso real de COVID de la U2).


## 4. Preparamos la comparación (misma muestra para los dos)

Para una comparación **justa**, damos a ambos modelos **los mismos datos**. Además, como TabFM lee el entrenamiento *como contexto* y su memoria crece con el nº de filas, tomamos una **submuestra** (con GPU gratuita, unos pocos miles de filas van sobrados). Los dos se evalúan sobre el **mismo** conjunto de test.


In [ ]:
from sklearn.model_selection import train_test_split

objetivo = "evento_cv"
cat = ["sexo","tabaquismo","actividad_fisica"]
feats = num + ["antecedentes_familiares","diabetes"] + cat
X, y = df[feats].copy(), df[objetivo].copy()

# submuestra estratificada: 5000 para entrenar/contexto, 2000 para test (mismos para ambos)
X_pool, X_test, y_pool, y_test = train_test_split(X, y, test_size=2000, train_size=5000,
                                                  random_state=42, stratify=y)
print("Contexto/entrenamiento:", X_pool.shape, "| Test:", X_test.shape)
print("Prevalencia de evento en test:", round(y_test.mean(), 3))

## 5. Rival 1 — **XGBoost bien afinado**

El estándar de oro en datos tabulares. Le damos el trato completo: preprocesado (escalado + *one-hot* de las categóricas) y **búsqueda de hiperparámetros** con validación cruzada. Es el trabajo que TabFM promete ahorrarnos.


In [ ]:
# XGBoost si está disponible; si no, usamos Gradient Boosting de scikit-learn (equivalente) como respaldo.
try:
    import xgboost  # en Colab suele estar; si no: !pip install xgboost
    from xgboost import XGBClassifier
    base = XGBClassifier(eval_metric="logloss", tree_method="hist", random_state=0)
    nombre_modelo = "XGBoost"
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    base = HistGradientBoostingClassifier(random_state=0)
    nombre_modelo = "Gradient Boosting (respaldo)"
print("Usaremos:", nombre_modelo)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score

pre = ColumnTransformer([("num", StandardScaler(), num + ["antecedentes_familiares","diabetes"]),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
pipe = Pipeline([("pre", pre), ("clf", base)])

# rejilla pequeña (para que corra rápido); en un proyecto real sería más amplia
if nombre_modelo.startswith("XGBoost"):
    grid = {"clf__n_estimators":[300,600], "clf__max_depth":[3,4,6],
            "clf__learning_rate":[0.03,0.1], "clf__subsample":[0.8,1.0]}
else:
    grid = {"clf__max_iter":[300,600], "clf__max_depth":[None,4,6], "clf__learning_rate":[0.03,0.1]}

busqueda = RandomizedSearchCV(pipe, grid, n_iter=8, scoring="roc_auc",
                              cv=StratifiedKFold(3, shuffle=True, random_state=0),
                              random_state=0, n_jobs=-1)
busqueda.fit(X_pool, y_pool)
xgb_auc = roc_auc_score(y_test, busqueda.predict_proba(X_test)[:,1])
print("Mejores hiperparámetros:", {k.replace('clf__',''):v for k,v in busqueda.best_params_.items()})
print(f"{nombre_modelo} afinado  ->  AUC en test = {xgb_auc:.3f}")

**Lo que vemos:** tras la búsqueda de hiperparámetros, el AUC ronda **~0,84**. Ha requerido preprocesado y varias pruebas cruzadas. Este es el listón que TabFM debe igualar… **sin nada de eso**.


## 6. Rival 2 — **TabFM en *zero-shot***

Ahora lo interesante. TabFM **no se entrena** sobre nuestros datos: lee las filas de entrenamiento **como contexto** y predice el test en **una sola pasada**. No hay hiperparámetros que ajustar, ni *one-hot* que preparar (acepta columnas mixtas tal cual).

Primero comprobamos si hay **GPU**; TabFM la necesita.


In [ ]:
# ¿Tenemos GPU? TabFM la requiere (en Colab: Entorno de ejecución -> GPU).
TABFM_OK = False
try:
    import torch
    TABFM_OK = torch.cuda.is_available()
except Exception:
    TABFM_OK = False
print("GPU disponible para TabFM:", TABFM_OK)
if not TABFM_OK:
    print(">> Si estás en Colab, activa la GPU y vuelve a ejecutar. Sin GPU se omiten las celdas de TabFM.")

Instalamos TabFM **solo si hay GPU** (clona el repo oficial e instala el *backend* PyTorch). La primera vez tarda un par de minutos; los pesos se descargan solos desde Hugging Face.


In [ ]:
import os, sys, subprocess
if TABFM_OK:
    if not os.path.isdir("tabfm"):
        subprocess.run(["git","clone","--depth","1","-q",
                        "https://github.com/google-research/tabfm.git"], check=False)
    subprocess.run([sys.executable,"-m","pip","install","-q","-e","./tabfm[pytorch]"], check=False)
    print("TabFM instalado (o ya estaba).")
else:
    print("Sin GPU: se omite la instalación de TabFM.")

Y ahora, el momento *zero-shot*: cargamos el modelo preentrenado, le pasamos el **contexto** (nuestras filas de entrenamiento **sin preprocesar**, con sus categorías en texto) y predecimos el test. Fíjate en lo corto que es el código comparado con la sección de XGBoost.


In [ ]:
tabfm_auc = None
if TABFM_OK:
    try:
        from tabfm import TabFMClassifier
        from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

        modelo_fm = tabfm_v1_0_0.load()            # descarga/carga los pesos preentrenados
        clf_fm = TabFMClassifier(model=modelo_fm)

        # TabFM acepta columnas mixtas (números + categorías en texto) SIN one-hot ni escalado manual
        clf_fm.fit(X_pool, y_pool)                 # "fit" solo prepara codificadores; NO entrena pesos
        p_fm = clf_fm.predict_proba(X_test)[:, 1]  # predicción en una sola pasada (in-context learning)

        tabfm_auc = roc_auc_score(y_test, p_fm)
        print(f"TabFM (zero-shot, sin ajustar nada)  ->  AUC en test = {tabfm_auc:.3f}")
    except Exception as e:
        print("No se pudo ejecutar TabFM en este entorno:", repr(e))
        print("Comprueba que estás en Colab con GPU y que la instalación terminó bien.")
else:
    print("TabFM omitido (sin GPU). Ábrelo en Colab con GPU para ver el resultado real.")

## 7. Cara a cara

Comparamos el AUC de los dos enfoques sobre el **mismo test**. Recuerda la diferencia de esfuerzo: XGBoost ha necesitado preprocesado + búsqueda de hiperparámetros; TabFM, **tres líneas y una pasada**.


In [ ]:
import matplotlib.pyplot as plt

resultados = {nombre_modelo + "\n(afinado)": xgb_auc}
if tabfm_auc is not None:
    resultados["TabFM\n(zero-shot)"] = tabfm_auc

print("AUC en test:")
for k, v in resultados.items():
    print(f"  {k.splitlines()[0]:22s}: {v:.3f}")

if len(resultados) == 2:
    fig, ax = plt.subplots(figsize=(6,4))
    nombres = list(resultados); valores = list(resultados.values())
    barras = ax.bar(nombres, valores, color=["#0B2447","#0E8F86"])
    ax.set_ylim(0.5, 0.95); ax.set_ylabel("AUC en test"); ax.set_title("Zero-shot vs. afinado")
    for b, v in zip(barras, valores):
        ax.text(b.get_x()+b.get_width()/2, v+0.005, f"{v:.3f}", ha="center", fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("\n(El gráfico comparativo aparece cuando ejecutas TabFM en Colab con GPU.)")

**Cómo leerlo.** Si TabFM iguala o supera a un XGBoost afinado **sin ajustar un solo hiperparámetro**, el mensaje es potente: para muchas tablas, el modelo fundacional da un resultado excelente **de fábrica**, y te ahorra horas de trabajo.

No siempre ganará (depende del problema, del tamaño y de la calidad de los datos), pero como **punto de partida sin esfuerzo** es difícil de batir. Lo honesto es probar ambos y quedarte con el que mejor rinda **en tu caso**, con las métricas de la U3.


## 8. Qué hemos aprendido

- **TabFM predice en *zero-shot***: sin entrenar ni ajustar hiperparámetros, leyendo el entrenamiento como **contexto** y prediciendo en una pasada. Compite con un **XGBoost afinado** con una fracción del esfuerzo.
- Acepta **columnas mixtas** (números + categorías) sin preprocesado manual: menos *feature engineering*.
- **Pero no hace magia con los datos:** la **limpieza y el EDA siguen siendo nuestros** (lo vimos partiendo de `pacientes_sucio.csv`).
- **Avisos:** necesita **GPU**; los **pesos son de licencia no comercial**; es una **caja negra** (menos interpretable que la logística de la U4); y en salud, como siempre, **hay que validarlo** antes de usarlo con pacientes.

> 🤖 **Con un asistente de IA** bastaría con pedir: *"Limpia pacientes_sucio.csv, compara un XGBoost afinado con TabFM en zero-shot para predecir evento_cv y dime cuál gana por AUC, con una partición estratificada"*. Tú aportas el **criterio** (qué comparar, qué métrica, validar); la herramienta, la velocidad.

Con esto cerramos la unidad de **modelos fundacionales**: los hay para imagen, para texto… y ahora también **para tablas**. La conclusión del curso se confirma: **cada vez entrenamos menos desde cero y elegimos, integramos y evaluamos más**.
